In [94]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import argrelextrema
from tabulate import tabulate
import ace_tools as tools
import warnings


ModuleNotFoundError: No module named 'ace_tools'

In [78]:
class Feature_Calculation:
    def __init__(self,data):
        self.data = data

    def Feature(self,window=20, num_std=2, calc_relative_width=False,RSI_period=30):
        delta = self.data['close'].diff()

        # 分别获取上涨和下跌
        gain = delta.where(delta > 0, 0.0)
        loss = -delta.where(delta < 0, 0.0)

        # Wilder's EMA 使用 alpha = 1/period
        # 首次计算简单平均值
        avg_gain = gain[:RSI_period].mean()
        avg_loss = loss[:RSI_period].mean()

        # 准备存放 RSI 的序列
        rsi_series = pd.Series(index=self.data.index, dtype=float)  # 使用 pd.Series 创建序列

        # 第一个 RSI 值
        rsi_series.iloc[RSI_period] = 100 - (100 / (1 + (avg_gain / avg_loss)))

        # 使用 Wilder 的平滑方法计算后续值
        for i in range(RSI_period + 1, len(self.data)):
            avg_gain = ((avg_gain * (RSI_period - 1)) + gain.iloc[i]) / RSI_period
            avg_loss = ((avg_loss * (RSI_period - 1)) + loss.iloc[i]) / RSI_period
            if avg_loss != 0:
                rs = avg_gain / avg_loss
                rsi_series.iloc[i] = 100 - (100 / (1 + rs))
            else:
                rsi_series.iloc[i] = 100

        self.data['rsi'] = rsi_series
        #print("当前RSI值：", self.data.tail(1)['rsi'])
        
        """
        计算布林带 (Bollinger Bands) 及布林带宽度。
        
        参数：
        - data: 包含 'close' 列的 Pandas DataFrame
        - window: 移动平均窗口大小，默认是 20
        - num_std: 标准差倍数，默认是 2
        - calc_relative_width: 是否计算相对宽度, 默认为 False
        
        返回：
        - 带有布林带新增列的 Pandas DataFrame:
        'BB_Middle', 'BB_Upper', 'BB_Lower', 'BB_Width', (可选) 'BB_Ratio'
        """
        # 计算中轨（简单移动平均）
        self.data['BB_Middle'] = self.data['close'].rolling(window=window).mean()
        
        # 计算标准差
        self.data['BB_Std'] = self.data['close'].rolling(window=window).std()
        
        # 计算上轨和下轨
        self.data['BB_Upper'] =  self.data['BB_Middle'] + num_std *  self.data['BB_Std']
        self.data['BB_Lower'] =  self.data['BB_Middle'] - num_std *  self.data['BB_Std']
        
        # 计算布林带绝对宽度
        self.data['BB_Width'] =  self.data['BB_Upper'] -  self.data['BB_Lower']
        
        # 如果需要，计算布林带相对宽度
        if calc_relative_width:
             self.data['BB_Ratio'] =  self.data['BB_Width'] /  self.data['BB_Middle']
        
        
    # def identify_support_resistance(self):
    #     last_close = self.data['close'].iloc[-1]  # 最新的收盘价
    #     last_rsi = self.data['rsi'].iloc[-1]  # 最新的RSI值
    #     last_upper_band = self.data['BB_Upper'].iloc[-1]  # 最新的布林带上轨
    #     last_lower_band = self.data['BB_Lower'].iloc[-1]  # 最新的布林带下轨
        
    #     support = None
    #     resistance = None
        
    #     # 通过布林带识别支撑和阻力
    #     if last_close <= last_lower_band:
    #         support = last_lower_band
    #     if last_close >= last_upper_band:
    #         resistance = last_upper_band
        
    #     # 通过RSI识别支撑和阻力
    #     if last_rsi <= 30:
    #         support = 'RSI 超卖：可能支撑'
    #     if last_rsi >= 70:
    #         resistance = 'RSI 超买：可能阻力'
    #     print(support, resistance)
    #     return support, resistance
    def find_local_extrema(self, order=10):
        """
        查找局部极大值（高点）和局部极小值（低点）
        :param order: 局部极值的查找窗口，默认为5
        :return: 返回局部极大值和局部极小值的索引
        """
        # 查找局部最大值（高点）
        local_max = argrelextrema(self.data['close'].values, np.greater, order=order)[0]
        # 查找局部最小值（低点）
        local_min = argrelextrema(self.data['close'].values, np.less, order=order)[0]

        return local_max, local_min
    def resample_data(self, period):
        """
        重采样数据到指定周期
        :param period: 重采样周期，如 '15T'（15分钟）、'30T'（30分钟）、'1H'（1小时）、'4H'（4小时）
        :return: 重采样后的数据
        """
        self.data.index = pd.to_datetime(self.data.index)
        resampled_data = self.data.resample(period).agg({
            'open': 'first',
            'high': 'max',
            'low': 'min',
            'close': 'last',
            'volume': 'sum'
        })
        return resampled_data.dropna()
   
### **2. 改进后的代码**


    def identify_support_resistance(self):
        """
        通过布林带、RSI、局部极点和移动平均线识别支撑位和阻力位
        :return: 包含支撑位和阻力位的字典，区分不同方法的结果
        """
        last_close = self.data['close'].iloc[-1]  # 最新的收盘价
        last_rsi = self.data['rsi'].iloc[-1]  # 最新的 RSI 值
        last_upper_band = self.data['BB_Upper'].iloc[-1]  # 最新的布林带上轨
        last_lower_band = self.data['BB_Lower'].iloc[-1]  # 最新的布林带下轨
        
        # 初始化支撑位和阻力位
        support_bb = None  # 布林带支撑位
        resistance_bb = None  # 布林带阻力位
        support_rsi = None  # RSI 支撑位
        resistance_rsi = None  # RSI 阻力位
        support_local = None  # 局部极点支撑位
        resistance_local = None  # 局部极点阻力位
        support_ma = None  # MA 支撑位
        resistance_ma = None  # MA 阻力位
        
        # 通过布林带识别支撑和阻力
        if last_close <= last_lower_band:
            support_bb = last_lower_band
        if last_close >= last_upper_band:
            resistance_bb = last_upper_band
        
        # 通过 RSI 识别支撑和阻力
        if last_rsi <= 30:
            support_rsi = 'RSI 超卖：可能支撑'
        if last_rsi >= 70:
            resistance_rsi = 'RSI 超买：可能阻力'
        
        # 通过局部极点识别支撑和阻力
        local_max, local_min = self.find_local_extrema(order=5)  # 查找局部极值
        if local_min.size > 0:  # 如果存在局部极小值
            support_local = self.data['close'].iloc[local_min[-1]]  # 最近的局部低点
        if local_max.size > 0:  # 如果存在局部极大值
            resistance_local = self.data['close'].iloc[local_max[-1]]  # 最近的局部高点
        
        # 通过移动平均线识别支撑和阻力
        ma_window = 20  # 移动平均线窗口大小
        ma = self.data['close'].rolling(window=ma_window).mean().iloc[-1]  # 最新的 MA 值
        support_ma = ma * 0.98  # MA 向下偏移 2% 作为支撑位
        resistance_ma = ma * 1.02  # MA 向上偏移 2% 作为阻力位
        
        # 打印结果
        print(f"布林带支撑位: {support_bb}, 布林带阻力位: {resistance_bb}")
        print(f"RSI 支撑位: {support_rsi}, RSI 阻力位: {resistance_rsi}")
        print(f"局部极点支撑位: {support_local}, 局部极点阻力位: {resistance_local}")
        print(f"MA 支撑位: {support_ma}, MA 阻力位: {resistance_ma}")
        
        # 返回结果
        return {
            'support_bb': support_bb,
            'resistance_bb': resistance_bb,
            'support_rsi': support_rsi,
            'resistance_rsi': resistance_rsi,
            'support_local': support_local,
            'resistance_local': resistance_local,
            'support_ma': support_ma,
            'resistance_ma': resistance_ma
        }

    def calculate_for_multiple_periods(self, periods=['15T', '30T', '1H', '4H']):
        """
        计算多个周期的支撑位和阻力位
        :param periods: 周期列表，如 ['15T', '30T', '1H', '4H']
        :return: 包含各周期支撑位和阻力位的字典
        """
        results = {}
        for period in periods:
            resampled_data = self.resample_data(period)
            if not resampled_data.empty:
                # 在每个周期数据上计算 RSI 和支撑/阻力
                feature_calc = Feature_Calculation(resampled_data)
                feature_calc.Feature(RSI_period=14)  # 可以调整 RSI 的周期
                sr=feature_calc.identify_support_resistance()
                results[period] = {'support_bb': sr['support_bb'], 'resistance_bb': sr['resistance_bb'],'support_rsi':sr['support_rsi'],'resistance_rsi':sr['resistance_rsi'],'support_local':sr['support_local'],'resistance_local':sr['resistance_local'],'support_ma':sr['support_ma'],'resistance_ma':sr['resistance_ma']}
        for period in periods:
            print(f"周期: {period}")
            print(f"支撑位: {results[period]['support_bb']}, 阻力位: {results[period]['resistance_bb']}")
            print(f"RSI 支撑位: {results[period]['support_rsi']}, RSI 阻力位: {results[period]['resistance_rsi']}")
            print(f"局部极点支撑位: {results[period]['support_local']}, 局部极点阻力位: {results[period]['resistance_local']}")
            print(f"MA 支撑位: {results[period]['support_ma']}, MA 阻力位: {results[period]['resistance_ma']}")
        return results
    def prepare_data_for_ml(self):
        """
        准备数据集用于机器学习模型训练，构建特征和标签
        :return: 特征数据和标签数据
        """
        # 构造特征列
        self.data['BB_Width'] = self.data['BB_Upper'] - self.data['BB_Lower']
        self.data['Distance_to_Upper'] = (self.data['close'] - self.data['BB_Upper']) / self.data['BB_Width']
        self.data['Distance_to_Lower'] = (self.data['BB_Lower'] - self.data['close']) / self.data['BB_Width']
        self.data['RSI'] = self.data['rsi']
        
        # 创建其他可能的特征列（如移动平均线、MACD等）
        self.data['MA20'] = self.data['close'].rolling(window=20).mean()
        self.data['MACD'] = self.data['close'].ewm(span=12).mean() - self.data['close'].ewm(span=26).mean()
        
        # 标签：当收盘价突破布林带的上下轨时设为1，否则为0
        self.data['Label'] = 0  # 默认是0
        self.data.loc[self.data['close'] > self.data['BB_Upper'], 'Label'] = 1  # 收盘价突破上轨
        self.data.loc[self.data['close'] < self.data['BB_Lower'], 'Label'] = -1  # 收盘价突破下轨

        # 删除空值
        self.data = self.data.dropna(subset=['BB_Width', 'Distance_to_Upper', 'Distance_to_Lower', 'RSI', 'MA20', 'MACD'])
        
        # 特征和标签分开
        X = self.data[['BB_Width', 'Distance_to_Upper', 'Distance_to_Lower', 'RSI', 'MA20', 'MACD']]
        y = self.data['Label']  # 标签，1表示突破上轨，-1表示突破下轨，0表示未突破

        return X, y
    def train_ml_model(self):
        """
        训练机器学习模型，预测支撑位和阻力位突破的概率
        :return: 训练好的模型
        """
        # 获取特征数据和标签数据
        X, y = self.prepare_data_for_ml()

        # 划分训练集和测试集（80%训练，20%测试）
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # 创建并训练随机森林分类模型
        model = RandomForestClassifier(n_estimators=100, random_state=42)
        model.fit(X_train, y_train)

        # 预测测试集
        y_pred = model.predict(X_test)

        # 输出模型评估结果
        print(classification_report(y_test, y_pred))

        return model
    



# 计算 Wilder's RSI



In [79]:
df = pd.read_csv('data/init.csv')

df.set_index('timestamp', inplace=True)
df = df.sort_index()
realTime = Feature_Calculation(df)


In [ ]:
realTime.Feature()
realTime.identify_support_resistance()
a=realTime.calculate_for_multiple_periods()

布林带支撑位: None, 布林带阻力位: None
RSI 支撑位: None, RSI 阻力位: None
局部极点支撑位: 2917.29, 局部极点阻力位: 2920.13
MA 支撑位: 2860.89342, MA 阻力位: 2977.66458
布林带支撑位: None, 布林带阻力位: 2916.9040218374103
RSI 支撑位: None, RSI 阻力位: None
局部极点支撑位: 2903.22, 局部极点阻力位: 2921.28
MA 支撑位: 2849.29855, MA 阻力位: 2965.5964500000005


/var/folders/vf/f1yl8qs536s4hz0355b67w0c0000gn/T/ipykernel_52620/3817452240.py:108: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  resampled_data = self.data.resample(period).agg({
/var/folders/vf/f1yl8qs536s4hz0355b67w0c0000gn/T/ipykernel_52620/3817452240.py:108: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  resampled_data = self.data.resample(period).agg({


布林带支撑位: None, 布林带阻力位: 2914.1711606311114
RSI 支撑位: None, RSI 阻力位: RSI 超买：可能阻力
局部极点支撑位: 2899.18, 局部极点阻力位: 2908.9
MA 支撑位: 2847.79817, MA 阻力位: 2964.0348300000005
布林带支撑位: None, 布林带阻力位: None
RSI 支撑位: None, RSI 阻力位: RSI 超买：可能阻力
局部极点支撑位: 2900.88, 局部极点阻力位: 2908.9
MA 支撑位: 2843.07506, MA 阻力位: 2959.1189400000003
布林带支撑位: None, 布林带阻力位: 2916.333065282557
RSI 支撑位: None, RSI 阻力位: RSI 超买：可能阻力
局部极点支撑位: 2860.09, 局部极点阻力位: 2869.19
MA 支撑位: 2815.83302, MA 阻力位: 2930.76498
周期: 15T
支撑位: None, 阻力位: 2916.9040218374103
RSI 支撑位: None, RSI 阻力位: None
局部极点支撑位: 2903.22, 局部极点阻力位: 2921.28
MA 支撑位: 2849.29855, MA 阻力位: 2965.5964500000005
周期: 30T
支撑位: None, 阻力位: 2914.1711606311114
RSI 支撑位: None, RSI 阻力位: RSI 超买：可能阻力
局部极点支撑位: 2899.18, 局部极点阻力位: 2908.9
MA 支撑位: 2847.79817, MA 阻力位: 2964.0348300000005
周期: 1H
支撑位: None, 阻力位: None
RSI 支撑位: None, RSI 阻力位: RSI 超买：可能阻力
局部极点支撑位: 2900.88, 局部极点阻力位: 2908.9
MA 支撑位: 2843.07506, MA 阻力位: 2959.1189400000003
周期: 4H
支撑位: None, 阻力位: 2916.333065282557
RSI 支撑位: None, RSI 阻力位: RSI 超买：可能阻力
局部极点支撑位: 286

/var/folders/vf/f1yl8qs536s4hz0355b67w0c0000gn/T/ipykernel_52620/3817452240.py:108: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  resampled_data = self.data.resample(period).agg({
/var/folders/vf/f1yl8qs536s4hz0355b67w0c0000gn/T/ipykernel_52620/3817452240.py:108: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  resampled_data = self.data.resample(period).agg({


,open,high,low,close,volume,vwap,transactions,rsi,BB_Middle,BB_Std,BB_Upper,BB_Lower,BB_Width
timestamp,,,,,,,,,,,,,
2024-10-30 00:00:00,2774.81,2774.90,2774.70,2774.70,3,2774.8033,3,NaN,NaN,NaN,NaN,NaN,NaN
2024-10-30 00:01:00,2775.05,2775.28,2775.05,2775.28,2,2775.1650,2,NaN,NaN,NaN,NaN,NaN,NaN
2024-10-30 00:02:00,2775.49,2775.49,2775.05,2775.44,3,2775.3267,3,NaN,NaN,NaN,NaN,NaN,NaN
2024-10-30 00:03:00,2775.53,2775.53,2775.43,2775.43,2,2775.4800,2,NaN,NaN,NaN,NaN,NaN,NaN
2024-10-30 00:04:00,2775.42,2775.53,2775.34,2775.34,3,2775.4300,3,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-10 23:55:00,2919.29,2919.29,2918.74,2918.74,2,2919.0150,2,62.062229,2919.2550,0.953942,2921.162884,2917.347116,3.815768
2025-02-10 23:56:00,2918.45,2919.52,2918.45,2918.62,3,2918.8633,3,61.635452,2919.3040,0.889621,2921.083242,2917.524758,3.558483
2025-02-10 23:57:00,2918.41,2918.62,2918.26,2918.62,3,2918.4300,3,61.635452,2919.3470,0.829839,2921.006678,2917.687322,3.319356
